# **Mini-Project \#2: Analysis of Nigerian Internally Displaced Persons Sites**

---





# **Identifying Vulnerable Populations in Nigerian IDP Sites**

## **Context**

Internally Displaced Persons (IDP) sites in northern Nigeria host very different populations. Some are dominated by young adults and working-age families; others concentrate groups whose needs are easy to under-serve in a generic camp response — infants, the elderly, female-headed households.

For humanitarian planners, treating every site the same wastes resources and leaves the most fragile people underserved. The first step toward targeted aid is knowing **which sites concentrate vulnerable demographics, and where those sites are**.

## **Main question**

> **Is there a "weak", vulnerable sub-population we should care for specifically — and if so, is it geographically concentrated?**

Concretely, we want to answer:

1. Do IDP sites fall into distinct demographic profiles based on age/sex composition?
2. Does any profile look like a vulnerable group requiring tailored support?
3. Is that group spread across the country, or concentrated in one zone?


The rest of the notebook tests this hypothesis with compositional clustering (CLR + K-Means) and a chi-square test of geographic association.

## **Setup**



In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})
PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#3B8132"]

AGE_COLS = [
    "INFANTS MALE", "INFANTS FEMALE",
    "CHILDREN MALE", "CHILDREN FEMALE",
    "YOUTH MALE", "YOUTH FEMALE",
    "ADULT MALE", "ADULT FEMALE",
    "ELDERLY MALE", "ELDERLY FEMALE",
]

## **Load and Clean the Data**

**TODO:**
1. Load `data.xlsx` into a DataFrame.
2. Drop any spurious `Unnamed:` columns that Excel sometimes adds.
3. Fix three known data-entry bugs:
   - `STATE` value `"KADUNNA"` should be `"KADUNA"`.
   - Site `BO_S002` has a missing/wrong `LATITUDE` — set it to `13.018227`.
   - Site `AD_S002` has a missing/wrong `LONGITUDE` — set it to `12.31593`.
4. Coerce all ten age-group columns to numeric (invalid entries become `NaN`).
5. Drop rows where any age column is missing.
6. Sanity-check that the row-wise sum of age columns matches `TOTAL NUMBER OF IDPS`
   to within ±100 — drop rows that fail this check.

In [4]:
#df = pd.read_excel("/content/drive/MyDrive/UNIGE/Data_Mining/UN Project/20161019-sample-data.xlsx")
df = pd.read_excel("20161019-sample-data.xlsx")
df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])

# Fix data ingestion bugs
df["STATE"] = df["STATE"].replace({"KADUNNA": "KADUNA"})
df.loc[df["SITE ID"] == "BO_S002", "LATITUDE"]  = 13.018227
df.loc[df["SITE ID"] == "AD_S002", "LONGITUDE"] = 12.31593

# Coerce age columns to numeric
for c in AGE_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows with missing demographics, then keep only consistent totals
clean = df.dropna(subset=AGE_COLS).copy()
clean["row_sum"] = clean[AGE_COLS].sum(axis=1)
clean["total_delta"] = clean["TOTAL NUMBER OF IDPS"] - clean["row_sum"]
clean = clean[clean["total_delta"].abs() < 100].reset_index(drop=True)

print(f"Rows after cleaning: {len(clean)}")
clean.head()

Rows after cleaning: 74


,SURVEY ROUND,SITE ID,SITE NAME,STATE,LGA,WARD,STATUS,LATITUDE,LONGITUDE,SITE MANAGEMENT AGENCY (SMA),...,CHILDREN FEMALE,YOUTH MALE,YOUTH FEMALE,ADULT MALE,ADULT FEMALE,ELDERLY MALE,ELDERLY FEMALE,TOTAL NUMBER OF IDPS,row_sum,total_delta
0,8,AD_S001,NYSC CAMP ADAMAWA,ADAMAWA,GIREI,DAMARE,FORMAL,9.19090,12.26852,NEMA,...,315,131,89,311,338,7,8,1512,1512.0,0.0
1,8,AD_S002,DEEPER LIFE CAMP GROUND,ADAMAWA,YOLA SOUTH,NAMTARI,FORMAL,9.30064,12.31593,CHURCH,...,30,20,40,38,11,2,4,175,175.0,0.0
2,8,AD_S003,MALKOHI CAMP,ADAMAWA,YOLA SOUTH,NAMTARI,FORMAL,9.19653,12.38689,NEMA/SEMA,...,104,224,196,203,317,14,8,1273,1273.0,0.0
3,8,AD_S008,MALKOHI VILLAGE,ADAMAWA,YOLA SOUTH,NAMTARI,INFORMAL,9.19288,12.37603,NaN,...,48,180,253,213,211,0,0,1005,1005.0,0.0
4,8,AD_S005,EYN CHURCH VINIKILANG,ADAMAWA,GIREI,MODIRE/ VINIKILANG,INFORMAL,9.18183,12.28432,IRC,...,2,13,6,6,11,1,1,51,51.0,0.0


## **Task I. Compositional Data Analysis (CLR transform)**

Before clustering the sites, we'll apply the **Centered Log-Ratio (CLR)** transform
to each row of age-group counts.

For a vector $\mathbf{x} = (x_1, \dots, x_D)$ with strictly positive components,
the CLR transform is defined as

$$\text{clr}(\mathbf{x}) = \left(\ln\frac{x_1}{g(\mathbf{x})},\; \dots,\; \ln\frac{x_D}{g(\mathbf{x})}\right), \qquad g(\mathbf{x}) = \sqrt[D]{x_1 \cdots x_D}$$

where $g(\mathbf{x})$ is the **geometric mean** of the row. Each transformed
coordinate is the log of the ratio between a component and that row's geometric
mean — a measure of how over- or under-represented the component is compared
to the typical component at that site. Note that, by construction, each
transformed row sums to zero.

**TODO:** produce a DataFrame named `clr_data` containing the CLR-transformed
age-group counts, by following these steps:

1. Build a DataFrame `raw_counts` from the `AGE_COLS` of `clean`, adding a
   pseudo-count of `+1` to every cell (avoids `log(0)`).
2. Build `prop_adj` by converting each row of `raw_counts` to proportions
   (divide each row by its sum).
3. Compute `geom_mean`: a Series containing the geometric mean of each row
   of `prop_adj`.
4. Build `clr_data`: a DataFrame of the same shape as `prop_adj` containing
   the CLR-transformed values.

**Expected deliverable.** After running your code, the following must hold:

- `clr_data` is a `pandas.DataFrame`.
- `clr_data.shape == (len(clean), len(AGE_COLS))`.
- `clr_data.columns` matches `AGE_COLS` (same names, same order).
- Every row of `clr_data` sums to (approximately) zero.
- `clr_data` contains no `NaN` or infinite values.

A test cell below will verify these properties automatically.

In [ ]:
raw_counts = clean[AGE_COLS].copy()

# Add pseudo-counts to prevent ln(0) errors, then convert to proportions

raw_counts = raw_counts + 1
prop_adj = raw_counts.div(raw_counts.sum(axis=1), axis=0)

# Geometric mean of each row, then CLR transform

geom_mean = np.exp(np.log(prop_adj).mean(axis=1))
clr_data = np.log(prop_adj.div(geom_mean, axis=0))

print("CLR data shape:", clr_data.shape)
print("Per-row sum (should be ~0 by construction):", clr_data.sum(axis=1).abs().max())
clr_data.head()


### **Question 1. Why bother with all this?**

CLR is not part of the standard clustering preprocessing toolkit. For most
datasets, the usual recipe is much simpler: z-score each column (subtract its
mean, divide by its standard deviation) and feed the result into K-Means (as you did before!). No
logs, no geometric means, no row-wise transformations.

For example, suppose we instead wanted to cluster the same IDP sites using a
different set of features:

| Feature | Example value |
|---|---|
| Total site population | 4,200 |
| Distance to nearest hospital (km) | 12.4 |
| Annual rainfall (mm) | 850 |
| Years since site was established | 6 |

For *that* dataset, plain z-scoring before K-Means would be entirely
appropriate — and CLR would be the wrong tool.

**Q:** explain why the age-group dataset requires
something like CLR, while the dataset in the table above does not. Identify
the structural property that distinguishes the two cases, and describe what
goes wrong if you ignore it and run K-Means on raw age-group counts (or
proportions) directly.



In [ ]:
# RUN THIS CELL TO CHECK IF YOUR PREVIOUS CODE IS CORRECT
# These checks should all pass if `clr_data` was built correctly above.

# 1. Shape: one row per cleaned site, one column per age group
assert clr_data.shape == (len(clean), len(AGE_COLS)), \
    f"Expected shape ({len(clean)}, {len(AGE_COLS)}), got {clr_data.shape}"

# 2. No NaN or infinite values (would indicate log(0) wasn't handled)
assert np.isfinite(clr_data.values).all(), \
    "clr_data contains NaN or inf — did you forget the +1 pseudo-count?"

# 3. Each row must sum to zero (defining property of CLR)
row_sums = clr_data.sum(axis=1)
assert np.allclose(row_sums, 0, atol=1e-10), \
    f"Rows should sum to 0; max deviation is {row_sums.abs().max()}"

# 4. Spot-check one row against an independent recomputation from scratch
i = 0
raw_i = clean[AGE_COLS].iloc[i].values + 1
prop_i = raw_i / raw_i.sum()
expected_i = np.log(prop_i) - np.mean(np.log(prop_i))
assert np.allclose(clr_data.iloc[i].values, expected_i, atol=1e-10), \
    "Row 0 does not match an independent CLR computation"

# 5. Scale invariance: scaling a row's raw counts by any positive constant
#    must give the same CLR coordinates (up to numerical noise).
raw_i_scaled = raw_i * 137.0
prop_i_scaled = raw_i_scaled / raw_i_scaled.sum()
clr_scaled = np.log(prop_i_scaled) - np.mean(np.log(prop_i_scaled))
assert np.allclose(expected_i, clr_scaled, atol=1e-10), \
    "CLR should be invariant to overall scale of the row"

print("✅ All CLR checks passed.")
print(f"   • {clr_data.shape[0]} sites × {clr_data.shape[1]} age groups")
print(f"   • Max |row sum|: {row_sums.abs().max():.2e}")
print(f"   • Value range: [{clr_data.values.min():.2f}, {clr_data.values.max():.2f}]")

**A:** The age-group variables are compositional within each site. Ten counts describe parts of one whole population. Because those parts are
linked by the site total, increasing the share of one group necessarily lowers
the relative share of at least one other group.

If we ran K-Means directly on raw age-group counts, large camps would look far from small camps because they have more people. Using simple proportions fixes the camp size
problem, but proportions are still constrained to sum to 1, so ordinary
Euclidean distances can treat small forced changes across many categories as
real independent variation. CLR handles this by converting each age group into
a log ratio relative to the site's geometric mean, so clustering focuses on
which groups are over or under represented within a site.

The comparison dataset in the table is different. Population size, distance to
hospital, rainfall, and site age are separate measurements, not parts of a
single fixed total. For those ordinary numeric features, z-scoring is the right
kind of preprocessing because it puts variables with different units onto a
comparable scale without needing a log ratio transform.


# **Task II. Identifying Demographic Clusters**

We're now ready to cluster the IDP sites themselves: each site is represented
by its CLR-transformed age-group profile, so two sites end up close together
when their *demographic shapes* are similar i.e. **a similar mix of infants,
children, youth, adults, and elderly, regardless of how big each camp is**.
The clusters K-means finds are therefore groups of sites that share a common
demographic signature, which is exactly what we need to identify whether a
distinct vulnerable profile (e.g. an elderly-heavy one) exists in the data.


--------------------------------------------------------------------------------




### Choosing the Number of Clusters | Why silhouette score rather than the elbow method?

The classic elbow method plots within-cluster inertia (the sum of squared
distances from each point to its centroid) against $k$ (what we did during
the previous sessions) and looks for the "elbow", the value of $k$ where
adding more clusters stops giving a big reduction in inertia. The trouble is
that inertia decreases *monotonically* with $k$ no matter what (more clusters
always fit the data more tightly), so the curve is always smooth and bending;
identifying *the* elbow is a subjective, eyeball judgement that often has no
clear answer. The **silhouette score** sidesteps this by measuring something
qualitatively different: for each point, how close it is to its own cluster
compared to the nearest *other* cluster. It rewards both **cohesion** (tight
clusters) and **separation** (well-separated clusters), and crucially it can
*go down* when $k$ is wrong — too few clusters lump distinct groups together,
too many split genuine groups apart. That gives us a real maximum to pick out
objectively, rather than a curve to squint at.

### Definition

For a point $i$ assigned to cluster $C_I$, define:

$$a(i) = \frac{1}{|C_I| - 1} \sum_{\substack{j \in C_I \\ j \neq i}} d(i, j)
\qquad \text{(mean distance to other points in its own cluster — cohesion)}$$

$$b(i) = \min_{J \neq I} \; \frac{1}{|C_J|} \sum_{j \in C_J} d(i, j)
\qquad \text{(mean distance to points in the nearest other cluster — separation)}$$

The silhouette of point $i$ is then

$$s(i) = \frac{b(i) - a(i)}{\max\bigl(a(i),\; b(i)\bigr)} \;\in\; [-1,\, +1].$$

The overall silhouette score for a clustering is the average of $s(i)$ across
all points:

$$S = \frac{1}{n} \sum_{i=1}^{n} s(i).$$

Higher $S$ means clusters are simultaneously tight and well-separated.

## **TODO:**
1. Fit K-Means on the CLR-transformed data for $k \in \{2, 3, \dots, 7\}$.
2. Compute the **silhouette score** $S$ for each $k$.
3. Plot $S$ as a function of $k$, and decide which value of $k$ you should
   keep for the rest of the analysis. Justify your choice in one or two
   sentences.


## **HINTS**
Please use the follow methods :
- `silhouette_score`
- `.labels_` attribute from your K-Means instantiated object.


In [ ]:
### FIT K MEANS AND SAVES SILHOUETTE SCORE

sil_scores = []
ks = range(2, 8)
for k in ks:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=30).fit(clr_data)
    sil_scores.append(silhouette_score(clr_data, km_temp.labels_))

### PLOT SILHOUETTE SCORE OVER K ###

best_k = int(np.argmax(sil_scores)) + 2
print(f"Optimal K = {best_k}")
for k, s in zip(ks, sil_scores):
    marker = "  <-- best" if k == best_k else ""
    print(f"  k={k}: silhouette={s:.4f}{marker}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(ks), sil_scores, "o-", color=PALETTE[0])
ax.axvline(best_k, color=PALETTE[1], linestyle="--", alpha=0.6, label=f"Best k = {best_k}")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette score vs. k")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
sil_scores = []
ks = range(2, 8)
for k in ks:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=30).fit(clr_data)
    sil_scores.append(silhouette_score(clr_data, km_temp.labels_))

best_k = int(np.argmax(sil_scores)) + 2
print(f"Optimal K = {best_k}")
for k, s in zip(ks, sil_scores):
    marker = "  <-- best" if k == best_k else ""
    print(f"  k={k}: silhouette={s:.4f}{marker}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(ks), sil_scores, "o-", color=PALETTE[0])
ax.axvline(best_k, color=PALETTE[1], linestyle="--", alpha=0.6, label=f"Best k = {best_k}")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette score vs. k")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### **QUESTION 2: Pick Optimal K**

**Q:** Decide which value of $k$ you should
   keep for the rest of the analysis. Justify your choice in one or two
   sentences.

**Answer.** The silhouette score selects $k = 2$, so I will treat the
two cluster solution as the primary model, however, $k = 4$ has a
similar silhouette score and may reveal finer demographic subgroups, so I will
also fit a four cluster solution as a sensitivity analysis. Later, I can
compare whether $k = 4$ adds useful interpretation or simply splits the same
broad demographic pattern into smaller groups.


## **Task III. Fit the Final K-Means Model**

**TODO:**
1. Refit K-Means on the CLR data using the best $k$ found above (use
   `random_state=42` and `n_init=30` for reproducibility).
2. Attach the cluster labels back onto the cleaned DataFrame. (for now the labels are just 0 and 1). Basically just use `.labels_` attribute from the `Kmeans` object obtained after fitting and save it into a new column named `cluster` in `clean` Dataframe!

In [ ]:
### RUN K MEANS WITH OPTIMAL K AND SAVE THE LABELS

primary_k = best_k
sensitivity_k = 4

km_primary = KMeans(n_clusters=primary_k, random_state=42, n_init=30).fit(clr_data)
clean["cluster"] = km_primary.labels_

km_k4 = KMeans(n_clusters=sensitivity_k, random_state=42, n_init=30).fit(clr_data)
clean["cluster_k4"] = km_k4.labels_



print(f"Primary model cluster sizes (k={primary_k}):")
print(clean["cluster"].value_counts().sort_index())

print(f"\nSensitivity model cluster sizes (k={sensitivity_k}):")
print(clean["cluster_k4"].value_counts().sort_index())

## **Task IV. PCA Biplot for Visualization**

To visualize how the clusters are arranged we project the 10-dimensional
CLR data onto its first two principal components — the two orthogonal
directions in feature space along which the sites vary the most. Plotting
each site by its (PC1, PC2) coordinates gives a 2D scatter we can color by
cluster.

A scatter alone tells us *that* the clusters separate, but not *why*. To
answer that, we draw a **biplot**: on top of the scatter we overlay one
arrow per original variable (here, per age group), pointing in the direction
of that variable in PC space. An arrow that points strongly into one cluster
means that age group is over-represented in that cluster; an arrow with
length close to zero means the variable barely contributes to either
component.

The arrow coordinates are called **loadings**. Conceptually, the loading of
variable $j$ on component $k$ is the correlation between the original
variable and the principal component — so loadings live in $[-1, +1]$ and
have one $(x, y)$ pair per feature. Mechanically, sklearn gives you two
ingredients to build them:

- `pca.components_` — shape `(n_components, n_features)`. Row $k$ is the
  unit-length direction of component $k$ expressed in the original feature
  space. Each row has $\ell_2$ norm 1, so these are pure directions, not
  weighted by importance.
- `pca.explained_variance_` — shape `(n_components,)`. Entry $k$ is the
  variance captured along component $k$. Larger = that component matters
  more.

The loading combines the two: take the direction of each feature in PC space
and scale it by how much variance that PC explains, so that long arrows
correspond to features driving large amounts of variation. The final
`loadings` array should have shape `(n_features, n_components) = (10, 2)` —
one row per age group, two columns giving the arrow's $(x, y)$ tip.

**TODO:**
1. Fit a 2-component PCA on `clr_data`. Store the fitted object as `pca`
   and the projected coordinates as `pca_result` (shape `(n_sites, 2)`).
2. Build the `loadings` array of shape `(10, 2)`. Derive the formula
   yourself from the two ingredients above — the dimensional analysis
   tells you both *what to multiply* and *how to align the axes* (you will
   need a transpose).
3. The plotting code (scatter, arrows, axis labels with explained variance,
   title) is provided. Run it once you have `pca`, `pca_result`, and
   `loadings`.

In [ ]:
# === Task: PCA biplot of the CLR-transformed sites ===
#
# Goal: project the 10-dimensional CLR data down to 2D so we can see the
# clusters, AND overlay the original age-group variables as arrows to see
# WHICH variables drive the separation between clusters (= "biplot").
#
# You need to fill in 2 things, marked TODO_1 and TODO_2.

# ----- TODO_1 -----
# Fit a PCA with 2 components on `clr_data` and store the projected coords.
# Expected: `pca` is a fitted sklearn PCA object;
#           `pca_result` is an (n_sites, 2) numpy array.
pca = ...
pca_result = ...

# ----- TODO_2 -----
# Compute the biplot LOADINGS for each original age-group variable.
# A loading is the correlation between an original variable and a principal
# component. A common formula is:
#     loadings = (PCA components, transposed) * sqrt(explained variance of each PC)
# Expected shape: (n_features, 2)  →  one (x, y) arrow per age group.
loadings = ...


# ----- Everything below is provided -----
clean["PCA1"] = pca_result[:, 0]
clean["PCA2"] = pca_result[:, 1]

fig, ax = plt.subplots(figsize=(9, 6))

# Scatter sites, colored by cluster
for c in range(best_k):
    mask = clean["cluster"] == c
    ax.scatter(clean.loc[mask, "PCA1"], clean.loc[mask, "PCA2"],
               c=PALETTE[c], label=f"C{c}: {CLUSTER_NAMES[c]} (n={mask.sum()})",
               alpha=0.7, edgecolor="white")

# One red arrow per age group, from origin out to its loading
for i, feature in enumerate(AGE_COLS):
    ax.arrow(0, 0, loadings[i, 0]*2.5, loadings[i, 1]*2.5,
             color="r", alpha=0.5, head_width=0.05)
    label = feature.replace(" MALE", " (M)").replace(" FEMALE", " (F)")
    ax.text(loadings[i, 0]*2.7, loadings[i, 1]*2.7, label,
            color="black", fontsize=8, ha="center")

ax.set_xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Modern Compositional Clustering (CLR + PCA + K-Means)")
ax.axhline(0, color="grey", linestyle="--", alpha=0.3)
ax.axvline(0, color="grey", linestyle="--", alpha=0.3)
ax.legend()
plt.grid(alpha=0.2)
plt.savefig("01_modern_pca_clusters.png")
plt.show()

### **Interpretation Questions**

Look at your biplot and answer in 2–3 sentences each:

1. **Cluster identity.** Which age-group arrows point toward each cluster?
   What does that tell you about the demographic profile of each cluster?

2. **Variance explained.** What percentage of the total variation does the
   2D projection capture (PC1 + PC2)? Is what you see a faithful summary of
   the full 10-dimensional structure, or are you missing a lot?


**Back to the main question**.
You now have everything you need to answer the question we started with:

> *Is there a "weak", vulnerable sub-population we should care for
> specifically — and if so, is it geographically concentrated?*

Write a short conclusion (5–10 sentences) that addresses the following:

3. **Did a vulnerable profile emerge from the clustering?** Describe what
   each cluster looks like demographically, and whether one of them
   qualifies as a vulnerable sub-population.

## **Task V. Geographic Interpretation**

Identifying a vulnerable demographic profile is only operationally useful if we know **where** those sites are. If the vulnerable cluster is scattered uniformly across the country, the humanitarian response has to be national — every camp gets the same services, thinly spread. But if it concentrates in one zone, aid can be **localized**: send mobility aids, chronic-disease medication, and age-appropriate nutrition specifically to the region that actually hosts those sites, instead of diluting the budget across the whole operation.

**TODO:**

Produce a scatter plot of all sites using their `LONGITUDE` and `LATITUDE` as the x and y coordinates, with each point colored by its cluster label. This gives us a geographic view of where each demographic cluster is located across the country.

In [ ]:
### COMPLETE HERE ###


## **Questions**


1. **Is that profile geographically concentrated?** Combining the chi-square
   p-value, the map, and the stacked bar chart, in which zone (if any) is
   the vulnerable cluster over-represented? Quantify it — e.g. "X% of
   sites in zone Y belong to the vulnerable cluster, vs. Z% elsewhere."

2. **Operational recommendation.** Given your findings, what would you
   recommend to a humanitarian planner? Should aid for this vulnerable
   group be deployed nationally, or concentrated in a specific region?
   Be concrete about *what kind* of aid (mobility, chronic-disease care,
   nutrition, etc.) and *where*.

3. **Caveats.** Name at least two limitations of this analysis that the
   planner should keep in mind before acting on your recommendation.
   Examples to think about: the 2D PCA only captures part of the variance,
   the cluster boundaries are not crisp, the dataset only covers a subset
   of states, the age categories are coarse, the data is a snapshot in
   time, etc.

4. **Next steps.** If you had access to one extra piece of information or
   one more analytical step, what would it be and why?